# 01 — Exploratory Data Analysis & Ego-Graph Topology (`TV1-06`)

Notebook này phân tích thống kê dữ liệu giao dịch Ethereum (`split_report.json`), phân phối bậc đồ thị (Power-Law In/Out-Degree), phân phối 25 đặc trưng đa phương thức trên tập `test` (`1,794,129` nút) và trực quan hóa cấu trúc Ego-Network 1-hop của ví Phishing so với ví Bình thường.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import pyarrow.parquet as pq

ROOT = Path.cwd().resolve()
if not (ROOT / 'configs').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.schemas import ALL_FEATURE_COLS, GRAPH_TOPO_COLS, TABULAR_TX_COLS, TEMPORAL_STAT_COLS
print('Repository root:', ROOT)

## 1. Chronological Split & Label Registry Audit (`TV1-02` / `TV1-03`)

In [ ]:
split_report = json.loads((ROOT / 'artifacts/results/split_report.json').read_text(encoding='utf-8'))
leak_report = json.loads((ROOT / 'artifacts/results/leak_audit_report.json').read_text(encoding='utf-8'))

print('Cutoffs:', json.dumps(split_report['cutoffs'], indent=2))
print('Address Split Summary:', json.dumps(split_report['address_split_summary'], indent=2))
print('Leakage Audit Status:', leak_report.get('overall_passed', True))

## 2. Class Distribution & 25-Feature Summary on Held-Out Labeled Nodes

In [ ]:
dataset = ds.dataset(str(ROOT / 'data/processed/graphs/test/nodes.parquet'), format='parquet')
labeled_tbl = dataset.to_table(filter=ds.field('label').isin([0, 1]))
labeled_df = labeled_tbl.to_pandas()

print(f'Total labeled nodes in test snapshot: {len(labeled_df):,}')
print(labeled_df['label'].value_counts().rename({0: 'Verified Normal (0)', 1: 'Known Phishing (1)'}))

summary_stats = labeled_df.groupby('label')[['tx_sent_count', 'tx_recv_count', 'in_degree', 'out_degree', 'local_ego_density', 'burstiness_index']].median()
display(summary_stats)

## 3. Heavy-Tailed Degree & Feature Separation (Phishing vs. Normal)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for lbl, name, color in [(0, 'Verified Normal', '#10b981'), (1, 'Known Phishing', '#f43f5e')]:
    sub = labeled_df[labeled_df['label'] == lbl]
    axes[0].hist(sub['out_degree'], bins=30, alpha=0.65, label=name, color=color, density=True)
    axes[1].hist(sub['burstiness_index'], bins=30, alpha=0.65, label=name, color=color, density=True)

axes[0].set_title('Scaled Out-Degree Distribution (Normal vs. Phishing)')
axes[0].legend()
axes[1].set_title('Scaled Burstiness Index Distribution')
axes[1].legend()
plt.tight_layout()
plt.show()

## 4. 1-Hop Ego-Network Inspection on Sample Fixture

In [ ]:
from src.app.engine import get_engine

engine = get_engine()
phish_ego = engine.get_ego_network('0x096f226b05777751047d1caa8e06f350fa7c4677', max_edges=25)
norm_ego = engine.get_ego_network('0x2260fac5e5542a773aa44fbcfedf7c193bc2c599', max_edges=25)

print(f'Phishing Wallet 1-Hop: in={phish_ego.in_degree}, out={phish_ego.out_degree}, peers={phish_ego.unique_counterparties}, vol={phish_ego.total_volume_eth:.4f} ETH')
print(f'Normal WBTC Hub 1-Hop: in={norm_ego.in_degree}, out={norm_ego.out_degree}, peers={norm_ego.unique_counterparties}, vol={norm_ego.total_volume_eth:.4f} ETH')